# Archived initial pipeline\n\nHistorical thesis-development notebook; not the final reported configuration. This public notebook retains the thesis research implementation, but the clinical dataset, derived volumes, annotations, labels, identifiers, checkpoints, and executed outputs are not included.\n\n## Configuration\nSet the paths below only to data for which you have appropriate authorisation. Do not commit local paths or generated clinical outputs.\n

In [ ]:
from pathlib import Path\n\nDATA_ROOT = Path(\"/path/to/authorised/data\")\nOUTPUT_ROOT = Path(\"./outputs\")\nOUTPUT_ROOT.mkdir(parents=True, exist_ok=True)\n

In [ ]:
# Google Colab Drive import removed for the public copy.\n# Configure DATA_ROOT below instead of mounting a private drive.


In [ ]:
import os, re
from pathlib import Path
import pandas as pd
import numpy as np


EXPECTED_PATIENTS = 37
DR = str(DATA_ROOT)

MUCUS_ROOT  = f"{DR}/MUCUS"
MUCUS2_ROOT = f"{DR}/MUCUS_2"
OUT_ROOT    = f"{DR}/mucus_master_37_fast"
Path(OUT_ROOT).mkdir(parents=True, exist_ok=True)

MASTER_INDEX_PATH = f"{OUT_ROOT}/master_patient_index.csv"
LABELS_CSV_PATH   = f"{OUT_ROOT}/labels.csv"


# ---- labels from excel ----
ANNOT_XLSX = f"{MUCUS_ROOT}/Annotations(2).xlsx"
lab = pd.read_excel(ANNOT_XLSX)

def norm_col(s): return re.sub(r"\s+", " ", str(s)).strip().lower()
lab = lab.rename(columns={c: norm_col(c) for c in lab.columns})

pid_col = "paziente" if "paziente" in lab.columns else None
if pid_col is None:
    raise RuntimeError(f"Can't find patient column in {ANNOT_XLSX}. Columns: {list(lab.columns)}")

if "media" in lab.columns:
    label_col = "media"
else:
    op_cols = [c for c in lab.columns if "operatore" in c or "operator" in c]
    lab["media"] = lab[op_cols].astype(float).mean(axis=1)
    label_col = "media"

labels = lab[[pid_col, label_col]].copy()
labels.columns = ["patient_id", "label"]
labels["patient_id"] = labels["patient_id"].astype(str).str.extract(r"(\d+)", expand=False)
labels = labels.dropna(subset=["patient_id"]).copy()
labels["patient_id"] = labels["patient_id"].astype(int).astype(str).str.zfill(4)
labels["label"] = labels["label"].astype(float)
labels = labels.drop_duplicates("patient_id", keep="first").reset_index(drop=True)
labels.to_csv(LABELS_CSV_PATH, index=False)


# ---- patient folders ----
def extract_patient_id(name: str) -> str:
    m = re.search(r"(\d+)", name)
    return str(m.group(1)).zfill(4) if m else ""

rows = []
for root in [MUCUS_ROOT, MUCUS2_ROOT]:
    for d in Path(root).iterdir():
        if d.is_dir():
            pid = extract_patient_id(d.name)
            if pid:
                rows.append({"patient_id": pid, "patient_dir": str(d), "source_root": root})

df_pat = pd.DataFrame(rows).sort_values("patient_id").reset_index(drop=True)

dups = df_pat[df_pat.duplicated("patient_id", keep=False)]
if len(dups):
    raise RuntimeError("Duplicate patient_id found across roots:\n" +
                       dups[["patient_id","patient_dir"]].to_string(index=False))

if len(df_pat) != EXPECTED_PATIENTS:
    print(f"WARNING: Expected {EXPECTED_PATIENTS}, found {len(df_pat)}")

df_master = df_pat.merge(labels, on="patient_id", how="left")

missing = df_master[df_master["label"].isna()][["patient_id","patient_dir"]]
if len(missing):
    raise RuntimeError("Missing labels for some patients:\n" + missing.to_string(index=False))

df_master.to_csv(MASTER_INDEX_PATH, index=False)

print("Wrote:")
print(" -", MASTER_INDEX_PATH)
print(" -", LABELS_CSV_PATH)


In [ ]:

OUT_ROOT = str(DATA_ROOT)
df = pd.read_csv(f"{OUT_ROOT}/master_patient_index.csv")
df["patient_id"] = df["patient_id"].astype(str).str.zfill(4)


# --- pydicom ---
!pip -q install pydicom
import pydicom

SKIP_DIR_NAMES = {"__macosx", ".ipynb_checkpoints", "cache", "logs", "tmp", "temp"}
SKIP_FILE_EXT  = {".txt",".csv",".json",".xml",".pdf",".zip",".rar",".7z",".html",
                  ".doc",".docx",".ppt",".pptx",".xls",".xlsx"}

BAD_SERIES = ["scout","localizer","topogram","surview"]


def iter_files(root: Path):
    for p in root.rglob("*"):
        if p.is_dir():
            if p.name.lower() in SKIP_DIR_NAMES:
                continue
            continue
        if p.name.startswith("."):
            continue
        if p.suffix.lower() in SKIP_FILE_EXT:
            continue
        yield p


def try_read_dicom_header(fp: Path):
    try:
        ds = pydicom.dcmread(str(fp), stop_before_pixels=True, force=True)
        if getattr(ds, "SOPInstanceUID", None) is None and getattr(ds, "SeriesInstanceUID", None) is None:
            return None
        return ds
    except Exception:
        return None


def sample_valid_dicoms(patient_dir: str, max_dicoms=25, max_checks=8000):
    out = []
    for i, fp in enumerate(iter_files(Path(patient_dir))):
        if i >= max_checks:
            break
        ds = try_read_dicom_header(fp)
        if ds is not None:
            out.append(ds)
            if len(out) >= max_dicoms:
                break
    return out


def summarize_header(ds):
    def g(attr, default=None): return getattr(ds, attr, default)
    series_desc = g("SeriesDescription")
    series_desc_l = str(series_desc).lower() if series_desc is not None else ""
    return {
        "modality": g("Modality"),
        "manufacturer": g("Manufacturer"),
        "model": g("ManufacturerModelName"),
        "kernel": g("ConvolutionKernel"),
        "slice_thickness": g("SliceThickness"),
        "pixel_spacing": g("PixelSpacing"),
        "spacing_between": g("SpacingBetweenSlices"),
        "kvp": g("KVP"),
        "series_desc": series_desc,
        "looks_like_scout": any(k in series_desc_l for k in BAD_SERIES),
    }

rows = []
for _, r in df.iterrows():
    pid = r["patient_id"]
    pdir = r["patient_dir"]
    root = Path(pdir)

    exts = []
    total_files = 0
    for fp in iter_files(root):
        total_files += 1
        exts.append(fp.suffix.lower())

    if total_files == 0:
        rows.append({"patient_id": pid, "patient_dir": pdir, "status": "EXCLUDE", "reason": "EMPTY_FOLDER",
                     "total_files": 0, "jpg": 0, "png": 0, "dcm_ext": 0, "other_ext": 0})
        continue

    ext_ser = pd.Series(exts)
    n_jpg = int((ext_ser == ".jpg").sum() + (ext_ser == ".jpeg").sum())
    n_png = int((ext_ser == ".png").sum())
    n_dcm_ext = int((ext_ser == ".dcm").sum())
    n_other = int(total_files - n_jpg - n_png - n_dcm_ext)

    samp = sample_valid_dicoms(pdir, max_dicoms=25)
    if len(samp) == 0:
        reason = "NON_DICOM_JPG" if n_jpg > 0 and (n_dcm_ext == 0) else "NO_DICOM_OTHER"
        rows.append({"patient_id": pid, "patient_dir": pdir, "status": "EXCLUDE", "reason": reason,
                     "total_files": total_files, "jpg": n_jpg, "png": n_png, "dcm_ext": n_dcm_ext, "other_ext": n_other})
        continue

    hdf = pd.DataFrame([summarize_header(ds) for ds in samp])

    def pick(col):
        s = hdf[col].dropna()
        return s.iloc[0] if len(s) else np.nan

    rows.append({
        "patient_id": pid,
        "patient_dir": pdir,
        "status": "OK",
        "reason": "OK_DICOM_FOUND",
        "total_files": total_files,
        "jpg": n_jpg,
        "png": n_png,
        "dcm_ext": n_dcm_ext,
        "other_ext": n_other,
        "sampled_dicoms": len(samp),
        "modality": pick("modality"),
        "manufacturer": pick("manufacturer"),
        "model": pick("model"),
        "kernel": pick("kernel"),
        "slice_thickness": pick("slice_thickness"),
        "pixel_spacing": pick("pixel_spacing"),
        "spacing_between": pick("spacing_between"),
        "kvp": pick("kvp"),
        "example_series_desc": pick("series_desc"),
        "any_sample_looks_like_scout": bool(hdf["looks_like_scout"].fillna(False).any()),
    })

card = pd.DataFrame(rows)


# Hard overrides (based on manual verification)
card.loc[card["patient_id"] == \"REDACTED_PATIENT_ID\", ["status","reason"]] = ["EXCLUDE","EMPTY_FOLDER"]
card.loc[card["patient_id"] == \"REDACTED_PATIENT_ID\", ["status","reason"]] = ["EXCLUDE","NON_DICOM_JPG"]

card_path = f"{OUT_ROOT}/data_card_v2.csv"
card.to_csv(card_path, index=False)

excluded = card[card["status"] != "OK"][["patient_id","reason","total_files","jpg","png","dcm_ext","other_ext"]]
excluded_path = f"{OUT_ROOT}/excluded_patients.csv"
excluded.to_csv(excluded_path, index=False)


print("Wrote:")
print(" -", card_path)
print(" -", excluded_path)
print("\nCounts:\n", card["status"].value_counts().to_string())
print("\nExcluded reasons:\n", excluded["reason"].value_counts().to_string())



In [ ]:
# --- Label distribution + diagnostics ---

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np


OUT_ROOT = str(DATA_ROOT)

df_master = pd.read_csv(f"{OUT_ROOT}/master_patient_index.csv")
df_master["patient_id"] = df_master["patient_id"].astype(str).str.zfill(4)
df_master["label"] = df_master["label"].astype(float)

card = pd.read_csv(f"{OUT_ROOT}/data_card_v2.csv")
card["patient_id"] = card["patient_id"].astype(str).str.zfill(4)


# CT-usable cohort (status OK)
ok_ids = set(card.loc[card["status"] == "OK", "patient_id"].tolist())

df_ok = df_master[df_master["patient_id"].isin(ok_ids)].copy()

def summarize_and_plot(df, title_suffix):
    y = df["label"].to_numpy()

    print(f"\n--- {title_suffix} ---")
    print("n patients:", len(df))
    print("min/max:", float(np.min(y)), float(np.max(y)))
    print("mean/std:", float(np.mean(y)), float(np.std(y, ddof=1)))
    print("median:", float(np.median(y)))
    print("quantiles 10/25/75/90:", [float(np.quantile(y, q)) for q in [0.1, 0.25, 0.75, 0.9]])

    freq = df["label"].value_counts().sort_index()
    print("\nLabel frequency:")
    print(freq.to_string())

    plt.figure()
    if freq.shape[0] <= 20:
        plt.bar(freq.index.astype(str), freq.values)
        plt.xticks(rotation=45, ha="right")
    else:
        plt.hist(y, bins=10)

    plt.xlabel("Label (Median)")
    plt.ylabel("Count")
    plt.title(f"Label distribution — {title_suffix}")
    plt.tight_layout()
    plt.show()



# Label distribution for CT-usable cohort (training cohort)
summarize_and_plot(df_ok, "CT-usable patients only (data_card status==OK)")

# Print excluded patients + their labels (important documentation)
excluded = card[card["status"] != "OK"][["patient_id","reason"]].merge(
    df_master[["patient_id","label"]], on="patient_id", how="left"
).sort_values("patient_id")

print("\nExcluded patients (and their labels):")
display(excluded)


In [ ]:

OUT_ROOT = str(DATA_ROOT)

# Load master + data_card_v2 and keep only usable patients
df = pd.read_csv(f"{OUT_ROOT}/master_patient_index.csv")
df["patient_id"] = df["patient_id"].astype(str).str.zfill(4)

card = pd.read_csv(f"{OUT_ROOT}/data_card_v2.csv")
card["patient_id"] = card["patient_id"].astype(str).str.zfill(4)

ok_ids = set(card.loc[card["status"] == "OK", "patient_id"].tolist())
df = df[df["patient_id"].isin(ok_ids)].copy()

SKIP_DIR_NAMES = {"__macosx", ".ipynb_checkpoints", "cache", "logs", "tmp", "temp"}

def estimate_max_slices(patient_dir: str, min_files_in_dir: int = 30):
    root = Path(patient_dir)
    best = 0
    for d in root.rglob("*"):
        if not d.is_dir():
            continue
        if d.name.lower() in SKIP_DIR_NAMES:
            continue
        try:
            n = sum(1 for p in d.iterdir() if p.is_file() and not p.name.startswith("."))
        except Exception:
            continue
        if n >= min_files_in_dir:
            best = max(best, n)
    if best == 0:
        try:
            best = sum(1 for p in root.iterdir() if p.is_file() and not p.name.startswith("."))
        except Exception:
            best = 0
    return best

df["max_slices_est"] = df["patient_dir"].apply(estimate_max_slices)

table = (
    df[["patient_id", "max_slices_est"]]
    .sort_values(["max_slices_est", "patient_id"], ascending=[False, True])
    .reset_index(drop=True)
)

# Pretty describe()
desc = table["max_slices_est"].describe()
desc.loc[["count","min","25%","50%","75%","max"]] = desc.loc[["count","min","25%","50%","75%","max"]].astype(int)
desc.loc[["mean","std"]] = desc.loc[["mean","std"]].round(2)
print(desc.to_string())

display(table)

# Save
out_path = f"{OUT_ROOT}/max_slices_est_ct_usable.csv"
table.to_csv(out_path, index=False)
print("Saved:", out_path)




In [ ]:
# 00 - Setup paths + load usable cohort before preprocessing

import os
import pandas as pd
from pathlib import Path

OUT_ROOT = str(DATA_ROOT)
MASTER_PATH = f"{OUT_ROOT}/master_patient_index.csv"
CARD_V2_PATH = f"{OUT_ROOT}/data_card_v2.csv"

assert os.path.exists(MASTER_PATH), "Missing master_patient_index.csv"
assert os.path.exists(CARD_V2_PATH), "Missing data_card_v2.csv"

master = pd.read_csv(MASTER_PATH)
master["patient_id"] = master["patient_id"].astype(str).str.zfill(4)

card = pd.read_csv(CARD_V2_PATH)
card["patient_id"] = card["patient_id"].astype(str).str.zfill(4)

usable_ids = set(card.loc[card["status"]=="OK", "patient_id"])
df = master[master["patient_id"].isin(usable_ids)].copy().reset_index(drop=True)

print("Usable patients:", len(df))
print("Example rows:")
display(df.head(10))



In [ ]:
# 01 - Lock folds (patient-level) and save split file

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, KFold

SPLIT_PATH = f"{OUT_ROOT}/splits_5fold_locked.csv"

y = df["label"].astype(float).values
seed = 42
n_splits = 5

def try_bins(y, max_bins=5):
    # Try qcut bins, reduce bins until each bin has at least 2 samples
    for nb in range(max_bins, 1, -1):
        try:
            b = pd.qcut(y, q=nb, duplicates="drop").codes
            if np.min(np.bincount(b)) >= 2:
                return b, f"qcut_{nb}"
        except Exception:
            pass
    return None, "no_bins"

bins, bin_method = try_bins(y, max_bins=5)

fold = np.full(len(df), -1, dtype=int)

if bins is not None:
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for k, (_, test_idx) in enumerate(skf.split(np.zeros(len(df)), bins)):
        fold[test_idx] = k
    method = f"StratifiedKFold({bin_method})"
else:
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for k, (_, test_idx) in enumerate(kf.split(np.zeros(len(df)))):
        fold[test_idx] = k
    method = "KFold_fallback"

splits = df[["patient_id","label"]].copy()
splits["fold"] = fold
splits["seed"] = seed
splits["method"] = method
splits.to_csv(SPLIT_PATH, index=False)

print("Wrote:", SPLIT_PATH)
print("Method:", method)
print(splits["fold"].value_counts().sort_index().to_dict())


In [ ]:
# 03 - Build series_index.csv (all series per patient)

SERIES_INDEX_PATH = f"{OUT_ROOT}/series_index.csv"

SKIP_DIR_NAMES = {"__macosx", ".ipynb_checkpoints", "cache", "logs", "tmp", "temp"}
SKIP_FILE_EXT  = {".txt",".csv",".json",".xml",".pdf",".zip",".rar",".7z",".html",
                  ".doc",".docx",".ppt",".pptx",".xls",".xlsx"}

def iter_files(root: Path):
    for p in root.rglob("*"):
        if p.is_dir():
            if p.name.lower() in SKIP_DIR_NAMES:
                continue
            continue
        if p.name.startswith("."):
            continue
        if p.suffix.lower() in SKIP_FILE_EXT:
            continue
        yield p

def read_hdr(fp: Path):
    try:
        ds = pydicom.dcmread(str(fp), stop_before_pixels=True, force=True)
        if getattr(ds, "SeriesInstanceUID", None) is None:
            return None
        def g(a, d=None): return getattr(ds, a, d)
        return {
            "path": str(fp),
            "Modality": g("Modality"),
            "SeriesInstanceUID": g("SeriesInstanceUID"),
            "StudyInstanceUID": g("StudyInstanceUID"),
            "SeriesDescription": g("SeriesDescription"),
            "ImageType": "\\".join(g("ImageType", [])) if isinstance(g("ImageType", []),(list,tuple)) else g("ImageType"),
            "InstanceNumber": g("InstanceNumber"),
            "SliceThickness": g("SliceThickness"),
            "PixelSpacing": g("PixelSpacing"),
            "SpacingBetweenSlices": g("SpacingBetweenSlices"),
            "ConvolutionKernel": g("ConvolutionKernel"),
            "Manufacturer": g("Manufacturer"),
            "ManufacturerModelName": g("ManufacturerModelName"),
            "KVP": g("KVP"),
            "Rows": g("Rows"),
            "Columns": g("Columns"),
        }
    except Exception:
        return None

rows = []
for _, r in df.iterrows():
    pid = r["patient_id"]
    pdir = Path(r["patient_dir"])
    n_checked = 0
    for fp in iter_files(pdir):
        n_checked += 1
        h = read_hdr(fp)
        if h is not None:
            rows.append({"patient_id": pid, **h})
        # safety cap per patient to avoid pathological folders
        if n_checked >= 50000:
            break

raw = pd.DataFrame(rows)
if raw.empty:
    raise RuntimeError("No DICOM headers found. Check paths and permissions.")

def first_nonnull(s):
    s = s.dropna()
    return s.iloc[0] if len(s) else np.nan

series = (raw.groupby(["patient_id","SeriesInstanceUID"], as_index=False)
            .agg(
                n_files=("path","count"),
                modality=("Modality", first_nonnull),
                study_uid=("StudyInstanceUID", first_nonnull),
                series_desc=("SeriesDescription", first_nonnull),
                image_type=("ImageType", first_nonnull),
                rows=("Rows", first_nonnull),
                cols=("Columns", first_nonnull),
                slice_thickness=("SliceThickness", first_nonnull),
                pixel_spacing=("PixelSpacing", first_nonnull),
                spacing_between=("SpacingBetweenSlices", first_nonnull),
                kernel=("ConvolutionKernel", first_nonnull),
                kvp=("KVP", first_nonnull),
                manufacturer=("Manufacturer", first_nonnull),
                model=("ManufacturerModelName", first_nonnull),
            ))

series.to_csv(SERIES_INDEX_PATH, index=False)
print("Wrote:", SERIES_INDEX_PATH)
print("Patients with at least 1 series:", series["patient_id"].nunique(), "/", len(df))
display(series.head(10))


In [ ]:
## 04 - Select the “best” CT axial series per patient → selected_series.csv

SELECTED_SERIES_PATH = f"{OUT_ROOT}/selected_series.csv"

series = pd.read_csv(SERIES_INDEX_PATH)
series["patient_id"] = series["patient_id"].astype(str).str.zfill(4)

BAD = ["scout", "localizer", "topogram", "surview", "dose", "report"]
GOOD = ["lung", "chest", "thorax", "thoracic", "pulmo"]

def score_row(row):
    score = float(row.get("n_files", 0))
    desc = str(row.get("series_desc", "")).lower()
    ityp = str(row.get("image_type", "")).lower()

    if any(k in desc for k in BAD) or any(k in ityp for k in BAD):
        score -= 1e6
    if any(k in desc for k in GOOD):
        score += 500

    # slice thickness sanity
    try:
        th = float(row.get("slice_thickness"))
        if 0.3 <= th <= 3.0:
            score += 200
        elif th > 6:
            score -= 300
    except Exception:
        pass

    return score

# prefer CT modality
ct = series[series["modality"].astype(str).str.upper() == "CT"].copy()
if ct.empty:
    raise RuntimeError("No CT series found in series_index. Something is wrong.")

ct["score"] = ct.apply(score_row, axis=1)

best = (ct.sort_values(["patient_id","score"], ascending=[True, False])
          .groupby("patient_id", as_index=False)
          .head(1)
          .reset_index(drop=True))

best.to_csv(SELECTED_SERIES_PATH, index=False)
print("Wrote:", SELECTED_SERIES_PATH)
display(best[["patient_id","n_files","series_desc","slice_thickness","pixel_spacing","kernel","manufacturer"]].head(10))


In [ ]:
# 05 - Rebuild volumes with a fallback series strategy + SimpleITK

import numpy as np
import pandas as pd
from pathlib import Path
import pydicom
import SimpleITK as sitk

OUT_ROOT = str(DATA_ROOT)
VOL_DIR = f"{OUT_ROOT}/volumes_hu"
Path(VOL_DIR).mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = f"{OUT_ROOT}/volume_manifest_.csv"

# load inputs
df = pd.read_csv(f"{OUT_ROOT}/master_patient_index.csv")
df["patient_id"] = df["patient_id"].astype(str).str.zfill(4)

card = pd.read_csv(f"{OUT_ROOT}/data_card_v2.csv")
card["patient_id"] = card["patient_id"].astype(str).str.zfill(4)
usable_ids = set(card.loc[card["status"]=="OK", "patient_id"])
df = df[df["patient_id"].isin(usable_ids)].reset_index(drop=True)

# reuse iter_files + skips (same as earlier cells)
SKIP_DIR_NAMES = {"__macosx", ".ipynb_checkpoints", "cache", "logs", "tmp", "temp"}
SKIP_FILE_EXT  = {".txt",".csv",".json",".xml",".pdf",".zip",".rar",".7z",".html",
                  ".doc",".docx",".ppt",".pptx",".xls",".xlsx"}

def iter_files(root: Path):
    for p in root.rglob("*"):
        if p.is_dir():
            if p.name.lower() in SKIP_DIR_NAMES:
                continue
            continue
        if p.name.startswith("."):
            continue
        if p.suffix.lower() in SKIP_FILE_EXT:
            continue
        yield p

def read_uid_and_meta(fp: Path):
    try:
        ds = pydicom.dcmread(str(fp), stop_before_pixels=True, force=True)
        uid = getattr(ds, "SeriesInstanceUID", None)
        if uid is None:
            return None
        mod = str(getattr(ds, "Modality", "")).upper()
        desc = str(getattr(ds, "SeriesDescription", "")).lower()
        ityp = str(getattr(ds, "ImageType", "")).lower()
        th = getattr(ds, "SliceThickness", None)
        return uid, mod, desc, ityp, th
    except Exception:
        return None

BAD = ["scout","localizer","topogram","surview","dose","report"]
GOOD = ["lung","chest","thorax","thoracic","pulmo"]

def score_series(uid, meta, n_files):
    mod, desc, ityp, th = meta
    score = float(n_files)

    if mod != "CT":
        score -= 5e5

    if any(k in desc for k in BAD) or any(k in ityp for k in BAD):
        score -= 1e6

    if any(k in desc for k in GOOD):
        score += 500

    try:
        thv = float(th)
        if 0.3 <= thv <= 3.0:
            score += 200
        elif thv > 6:
            score -= 300
    except Exception:
        pass

    return score

def hu_from_sitk(file_list):
    # Read volume
    img = sitk.ReadImage(file_list)  # handles compressed pixel data well
    arr = sitk.GetArrayFromImage(img).astype(np.float32)  # (Z,H,W)

    # Apply slope/intercept from first slice header (header-only; safe)
    ds0 = pydicom.dcmread(file_list[0], stop_before_pixels=True, force=True)
    slope = float(getattr(ds0, "RescaleSlope", 1.0))
    intercept = float(getattr(ds0, "RescaleIntercept", 0.0))
    hu = arr * slope + intercept

    # spacing (x,y,z) in mm
    sx, sy, sz = img.GetSpacing()  # sitk gives (x,y,z)
    return hu.astype(np.int16), (sx, sy, sz)

manifest_rows = []

for _, row in df.iterrows():
    pid = row["patient_id"]
    pdir = Path(row["patient_dir"])

    # One pass: build uid -> file list + meta
    uid_files = {}
    uid_meta = {}
    for fp in iter_files(pdir):
        info = read_uid_and_meta(fp)
        if info is None:
            continue
        uid, mod, desc, ityp, th = info
        uid_files.setdefault(uid, []).append(str(fp))
        uid_meta.setdefault(uid, (mod, desc, ityp, th))

    if not uid_files:
        manifest_rows.append({"patient_id": pid, "status": "FAIL_NO_SERIES", "volume_path": ""})
        continue

    # Rank series by score
    ranked = []
    for uid, files in uid_files.items():
        meta = uid_meta.get(uid, ("","", "", None))
        ranked.append((score_series(uid, meta, len(files)), uid))
    ranked.sort(reverse=True)

    # Try series until success
    success = False
    last_err = ""
    for score, uid in ranked[:5]:  # try top-5 candidates
        files = uid_files[uid]
        # SimpleITK expects ordered slices; sort by InstanceNumber if possible
        try:
            # sort using InstanceNumber from header (fast)
            inst = []
            for f in files:
                ds = pydicom.dcmread(f, stop_before_pixels=True, force=True)
                inst.append((getattr(ds, "InstanceNumber", 0), f))
            inst.sort(key=lambda x: x[0])
            files_sorted = [f for _, f in inst]
        except Exception:
            files_sorted = files

        try:
            vol_hu, (sx, sy, sz) = hu_from_sitk(files_sorted)
            if vol_hu.shape[0] < 10:
                last_err = f"TOO_FEW_SLICES({vol_hu.shape[0]})"
                continue

            out_path = f"{VOL_DIR}/{pid}.npy"
            np.save(out_path, vol_hu)

            manifest_rows.append({
                "patient_id": pid,
                "status": "OK",
                "volume_path": out_path,
                "shape": str(tuple(vol_hu.shape)),
                "n_slices": int(vol_hu.shape[0]),
                "spacing_x_mm": float(sx),
                "spacing_y_mm": float(sy),
                "spacing_z_mm": float(sz),
                "used_series_uid": uid,
                "used_series_score": float(score),
                "modality": uid_meta[uid][0],
                "series_desc": uid_meta[uid][1],
            })
            success = True
            break
        except Exception as e:
            last_err = str(e)[:200]
            continue

    if not success:
        manifest_rows.append({
            "patient_id": pid,
            "status": "FAIL_ALL_SERIES",
            "volume_path": "",
            "error": last_err
        })

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(MANIFEST_PATH, index=False)

print("Wrote:", MANIFEST_PATH)
print("OK volumes:", int((manifest["status"]=="OK").sum()), "/", len(manifest))
display(manifest["status"].value_counts())



In [ ]:
import torch

!pip -q install pydicom SimpleITK
import pydicom
import SimpleITK as sitk
print("pydicom:", pydicom.__version__)
print("SimpleITK:", sitk.Version_VersionString())



!pip -q install gdcm pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg

In [ ]:
torch.cuda.is_available(), torch.cuda.get_device_name(0)

In [ ]:
# 06 - Build one unified training index (manifest + labels + folds)

import pandas as pd
import numpy as np


OUT_ROOT = str(DATA_ROOT)

MANIFEST_PATH = f"{OUT_ROOT}/volume_manifest_.csv"
SPLITS_PATH   = f"{OUT_ROOT}/splits_5fold_locked.csv"
MASTER_PATH   = f"{OUT_ROOT}/master_patient_index.csv"

man = pd.read_csv(MANIFEST_PATH)
man["patient_id"] = man["patient_id"].astype(str).str.zfill(4)
man = man.query("status == 'OK'").copy()

spl = pd.read_csv(SPLITS_PATH)
spl["patient_id"] = spl["patient_id"].astype(str).str.zfill(4)

master = pd.read_csv(MASTER_PATH)
master["patient_id"] = master["patient_id"].astype(str).str.zfill(4)
master["label"] = master["label"].astype(float)

df_index = (man.merge(master[["patient_id","label"]], on="patient_id", how="left")
              .merge(spl[["patient_id","fold"]], on="patient_id", how="left"))

assert df_index["label"].notna().all(), "Some labels missing after merge."
assert df_index["fold"].notna().all(), "Some folds missing after merge."
assert df_index["volume_path"].apply(lambda p: isinstance(p,str) and len(p)>0).all(), "Missing volume_path."
assert len(df_index) == 35, f"Expected 35 OK volumes; got {len(df_index)}"

df_index = df_index.sort_values("patient_id").reset_index(drop=True)

print("Training index rows:", len(df_index))
display(df_index[["patient_id","label","fold","n_slices","spacing_z_mm","series_desc"]].head(12))

INDEX_PATH = f"{OUT_ROOT}/train_index.csv"
df_index.to_csv(INDEX_PATH, index=False)
print("Saved:", INDEX_PATH)



In [ ]:
# Cell 07 — Preprocessing utilities (window + normalize + resize)

import numpy as np
import cv2

IMG_SIZE = 256

# Lung window defaults (good baseline)
WL = -600
WW = 1500
HU_MIN = WL - WW/2   # -1350
HU_MAX = WL + WW/2   # 150

def window_and_normalize(slice_hu: np.ndarray, hu_min=HU_MIN, hu_max=HU_MAX) -> np.ndarray:
    x = np.clip(slice_hu, hu_min, hu_max).astype(np.float32)
    x = (x - hu_min) / (hu_max - hu_min + 1e-6)   # -> ~[0,1]
    return x

def resize2d(x: np.ndarray, size=IMG_SIZE) -> np.ndarray:
    # x: float32, shape (H,W)
    x = cv2.resize(x, (size, size), interpolation=cv2.INTER_AREA)
    # numerical safety
    x = np.clip(x, 0.0, 1.0).astype(np.float32)
    return x

def preprocess_slice(slice_hu: np.ndarray) -> np.ndarray:
    x = window_and_normalize(slice_hu)
    x = resize2d(x)
    return x  # float32 (size,size) in [0,1]


In [ ]:
# Cell 08 — Representative slice sampling (cluster-based, per patient)
import numpy as np

# keep consistent with our earlier baseline for comparison
CENTER_FRACTION = 0.6
N_SLICES_REP = 150
FEAT_HW = 256

def _central_candidates(n_slices: int, center_fraction: float):
    if n_slices <= 0:
        return np.array([], dtype=int)
    if center_fraction < 1.0:
        margin = int((1 - center_fraction) * n_slices / 2)
        lo = max(margin, 0)
        hi = max(n_slices - margin, lo + 1)
    else:
        lo, hi = 0, n_slices
    return np.arange(lo, hi, dtype=int)

def _downsample_blockmean(x2d: np.ndarray, out_hw: int = 32):
    """
    Fast block-mean downsample. Best when H,W divisible by out_hw (256->32 works great).
    """
    H, W = x2d.shape
    if (H % out_hw == 0) and (W % out_hw == 0):
        fh, fw = H // out_hw, W // out_hw
        return x2d.reshape(out_hw, fh, out_hw, fw).mean(axis=(1, 3))

    sh = max(1, H // out_hw)
    sw = max(1, W // out_hw)
    y = x2d[::sh, ::sw][:out_hw, :out_hw]
    if y.shape != (out_hw, out_hw):
        out = np.zeros((out_hw, out_hw), dtype=y.dtype)
        out[:y.shape[0], :y.shape[1]] = y
        y = out
    return y

def _ensure_k_unique(idx: np.ndarray, candidates: np.ndarray, k: int, rng=None):
    """
    Ensure exactly k unique indices from candidates when len(candidates) >= k.
    """
    rng = np.random.default_rng() if rng is None else rng

    idx = np.unique(idx.astype(int))
    candidates = np.unique(candidates.astype(int))

    # safety: keep only indices inside candidates
    idx = idx[np.isin(idx, candidates)]

    if len(candidates) <= k:
        return np.sort(candidates)

    if len(idx) >= k:
        keep = np.linspace(0, len(idx) - 1, k).round().astype(int)
        return np.sort(idx[keep])

    need = k - len(idx)
    remaining = candidates[~np.isin(candidates, idx)]

    # random fill first
    if len(remaining) > 0:
        take = min(need, len(remaining))
        fill = rng.choice(remaining, size=take, replace=False)
        idx = np.unique(np.concatenate([idx, fill]))

    # uniform fill if still short (rare)
    if len(idx) < k:
        need = k - len(idx)
        base = np.linspace(0, len(candidates) - 1, need).round().astype(int)
        idx = np.unique(np.concatenate([idx, candidates[base]]))

    # final trim (if overshoot)
    if len(idx) > k:
        keep = np.linspace(0, len(idx) - 1, k).round().astype(int)
        idx = idx[keep]

    return np.sort(idx.astype(int))

def sample_slice_indices_representative(
    vol: np.ndarray,
    n_select: int = N_SLICES_REP,
    center_fraction: float = CENTER_FRACTION,
    rng=None,
    feat_hw: int = FEAT_HW,
):
    """
    Representative sampling via MiniBatchKMeans on cheap per-slice features.

    - Uses central portion of the scan (CENTER_FRACTION).
    - Builds cheap features by HU windowing + block-mean downsampling.
    - Clusters slices and picks the closest slice to each centroid.
    - Guarantees exactly n_select unique indices when possible.
    """
    rng = np.random.default_rng() if rng is None else rng
    Z = int(vol.shape[0])
    candidates = _central_candidates(Z, center_fraction)

    if len(candidates) == 0:
        return np.array([max(0, Z // 2)], dtype=int)

    if len(candidates) <= n_select:
        return candidates.astype(int)

    # cheap HU window + normalize
    x = vol[candidates].astype(np.float32)               # [M,H,W]
    x = np.clip(x, -1000.0, 400.0)
    x = (x + 1000.0) / 1400.0                           # ~[0,1]

    # features: downsample each slice then flatten
    feats = np.stack(
        [_downsample_blockmean(xi, out_hw=feat_hw).reshape(-1) for xi in x],
        axis=0
    )                                                   # [M,F]

    # standardize
    feats = feats - feats.mean(axis=0, keepdims=True)
    feats = feats / (feats.std(axis=0, keepdims=True) + 1e-6)

    k = int(min(n_select, feats.shape[0]))

    try:
        from sklearn.cluster import MiniBatchKMeans
        from sklearn.metrics import pairwise_distances_argmin_min

        km = MiniBatchKMeans(
            n_clusters=k,
            random_state=int(rng.integers(0, 1_000_000_000)),
            batch_size=512,
            n_init="auto",
            max_iter=200,
        )
        km.fit(feats)

        nearest, _ = pairwise_distances_argmin_min(km.cluster_centers_, feats)
        idx = candidates[nearest]                       # may contain duplicates

        idx = _ensure_k_unique(idx, candidates=candidates, k=k, rng=rng)
        return idx.astype(int)

    except Exception as e:
        print("[warn] representative sampling failed:", str(e))
        # safe deterministic-ish fallback: evenly spaced selection
        base = np.linspace(0, len(candidates) - 1, k).round().astype(int)
        return np.sort(candidates[base]).astype(int)


In [ ]:
# Cell 09A — PyTorch Dataset for 2D slice batches + patient-level label (representative)

!pip -q install torch torchvision

import torch
from torch.utils.data import Dataset
import pandas as pd
import numpy as np

class PatientSliceDataset(Dataset):
    def __init__(self, index_csv: str, fold: int, split: str, seed: int = 42):
        """
        split: 'train' or 'val'
        fold: which fold is validation
        """
        df = pd.read_csv(index_csv)
        df["patient_id"] = df["patient_id"].astype(str).str.zfill(4)
        df["label"] = df["label"].astype(float)
        df["fold"] = df["fold"].astype(int)

        if split == "train":
            self.df = df[df["fold"] != fold].reset_index(drop=True)
        else:
            self.df = df[df["fold"] == fold].reset_index(drop=True)

        self.split = split
        self.fold = int(fold)
        self.seed = int(seed)

        # cache representative indices so we don’t recluster every epoch
        # key includes fold/split/settings to avoid accidental reuse
        self.rep_cache = {}

    def _make_deterministic_rng(self, pid: str, tag: str):
        s = (hash(f"{pid}|fold{self.fold}|{tag}|seed{self.seed}") & 0xffffffff)
        return np.random.default_rng(s)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        pid = str(row["patient_id"]).zfill(4)
        y = float(row["label"])
        vol = np.load(row["volume_path"])  # (Z,H,W) HU int16
        n_slices = int(vol.shape[0])

        # --------------------------
        # Representative slice sampling (cached, deterministic)
        # --------------------------
        cache_key = (pid, self.split, self.fold, self.seed, N_SLICES_REP, CENTER_FRACTION)

        if cache_key not in self.rep_cache:
            rng = self._make_deterministic_rng(pid, "rep_train" if self.split == "train" else "rep_val")
            idx = sample_slice_indices_representative(vol, rng=rng)  # from Cell 08B
            self.rep_cache[cache_key] = idx

        idx = self.rep_cache[cache_key]

        if len(idx) == 0:
            idx = np.array([n_slices // 2], dtype=int)

        slices = [preprocess_slice(vol[int(j)]) for j in idx]  # each (S,S) float32 [0,1]
        x = np.stack(slices, axis=0)      # (K,S,S)
        x = x[:, None, :, :]              # (K,1,S,S)
        x = torch.from_numpy(x).float()
        y = torch.tensor(y).float()

        return x, y, pid


# quick sanity check
ds = PatientSliceDataset(index_csv=f"{OUT_ROOT}/train_index.csv", fold=0, split="train")
x, y, pid = ds[0]
print("Train example:", pid, "x:", tuple(x.shape), "y:", float(y))

ds_val = PatientSliceDataset(index_csv=f"{OUT_ROOT}/train_index.csv", fold=0, split="val")
x2, y2, pid2 = ds_val[0]
print("Val example:", pid2, "x:", tuple(x2.shape), "y:", float(y2))

print("Sampling mode: representative")



In [ ]:
import pandas as pd
import numpy as np

def rep_K_from_Z(Z, center_fraction=0.6, n_slices_rep=100):
    margin = int((1 - center_fraction) * Z / 2)
    candidates = max(0, Z - 2*margin)
    return min(n_slices_rep, candidates), candidates, margin

def debug_fold_split(index_csv, fold, center_fraction=0.6, n_slices_rep=100):
    df = pd.read_csv(index_csv)
    df["patient_id"] = df["patient_id"].astype(str).str.zfill(4)

    train = df[df["fold"] != fold].reset_index(drop=True)
    val   = df[df["fold"] == fold].reset_index(drop=True)

    for name, sub in [("train", train), ("val", val)]:
        print(f"\nFold {fold} — {name.upper()} — n_patients={len(sub)}")
        for _, r in sub.iterrows():
            pid = r["patient_id"]
            y   = float(r["label"])
            vol = np.load(r["volume_path"], mmap_mode="r")
            Z   = int(vol.shape[0])
            K, cand, margin = rep_K_from_Z(Z, center_fraction, n_slices_rep)
            print(f"  pid={pid}  y={y:.3f}  Z={Z:>4}  central_candidates={cand:>4}  K_rep={K:>3}")

# run for one fold
debug_fold_split(f"{OUT_ROOT}/train_index.csv", fold=4, center_fraction=CENTER_FRACTION, n_slices_rep=N_SLICES_REP)


In [ ]:
# Cell 10A) Download RadImageNet PyTorch pretrained weights

!pip -q install gdown

import os, zipfile
import gdown
from pathlib import Path

OUT_ROOT = str(DATA_ROOT)
Path(OUT_ROOT).mkdir(parents=True, exist_ok=True)

RADIMAGENET_URL = "https://drive.google.com/uc?id=1RHt2GnuOYlc_gcoTETtBDSW73mFyRAtR"
zip_path = f"{OUT_ROOT}/radimagenet_pytorch_models.zip"
dst_dir  = f"{OUT_ROOT}/radimagenet_weights"

if not os.path.exists(zip_path):
    gdown.download(RADIMAGENET_URL, zip_path, quiet=False)

Path(dst_dir).mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(dst_dir)

# show what we got
from glob import glob
files = sorted(glob(dst_dir + "/**/*", recursive=True))
print("Extracted files (first 50):")
for f in files[:50]:
    print(" -", f)
print("\nTotal extracted:", len(files))


In [ ]:
# 10B) - RadImageNet weight paths

import os

OUT_ROOT = str(DATA_ROOT)
RADIMAGENET_DIR = f"{OUT_ROOT}/radimagenet_weights/RadImageNet_pytorch"

RADIMAGENET_WEIGHTS = {
    "resnet50":    os.path.join(RADIMAGENET_DIR, "ResNet50.pt"),
    "densenet121": os.path.join(RADIMAGENET_DIR, "DenseNet121.pt"),
    "inceptionv3": os.path.join(RADIMAGENET_DIR, "InceptionV3.pt"),
}

# quick check
for k, p in RADIMAGENET_WEIGHTS.items():
    print(k, "->", p, "| exists:", os.path.exists(p))

In [ ]:
# 10C) — Model: flexible backbone (ImageNet or RadImageNet) + slice→patient pooling

import os
import torch
import torch.nn as nn
import torchvision.models as models

def _clean_state_dict(sd):
    """Strip common wrappers: {'state_dict':...}, 'module.' and return flat dict."""
    if isinstance(sd, dict) and "state_dict" in sd:
        sd = sd["state_dict"]
    out = {}
    for k, v in sd.items():
        if k.startswith("module."):
            k = k[len("module."):]
        out[k] = v
    return out

def _strip_prefix(sd, prefix):
    """Strip a prefix like 'backbone.' if present."""
    out = {}
    for k, v in sd.items():
        if k.startswith(prefix):
            out[k[len(prefix):]] = v
        else:
            out[k] = v
    return out

class SliceRegressor(nn.Module):
    """
    Works with:
      - resnet18 (ImageNet weights via torchvision)
      - resnet50 (ImageNet or RadImageNet .pt)
      - densenet121 (ImageNet or RadImageNet .pt)
      - inceptionv3 (ImageNet or RadImageNet .pt)
    """
    def __init__(self, backbone="resnet18", imagenet_pretrained=True, rad_weight_path=None):
        super().__init__()
        self.backbone_name = backbone

        # input is 1-channel; most pretrained nets expect 3
        self.in_proj = nn.Conv2d(1, 3, kernel_size=1, bias=False)

        # -----------------------
        # Build backbone
        # -----------------------
        if backbone == "resnet18":
            net = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if imagenet_pretrained else None)
            feat_dim = net.fc.in_features
            net.fc = nn.Identity()
            self.backbone = net

        elif backbone == "resnet50":
            net = models.resnet50(weights=models.ResNet50_Weights.DEFAULT if imagenet_pretrained else None)

            # sequential backbone: [conv1, bn1, relu, maxpool, layer1, layer2, layer3, layer4, avgpool, flatten]
            self.backbone = nn.Sequential(
                net.conv1,    # 0
                net.bn1,      # 1
                net.relu,     # 2
                net.maxpool,  # 3
                net.layer1,   # 4
                net.layer2,   # 5
                net.layer3,   # 6
                net.layer4,   # 7
                net.avgpool,  # 8
                nn.Flatten(1) # 9
            )
            feat_dim = 2048

        elif backbone == "densenet121":
            net = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT if imagenet_pretrained else None)
            feat_dim = net.classifier.in_features
            net.classifier = nn.Identity()
            self.backbone = net

        elif backbone == "inceptionv3":
            net = models.inception_v3(
                weights=models.Inception_V3_Weights.DEFAULT if imagenet_pretrained else None,
                aux_logits=False
            )
            feat_dim = net.fc.in_features
            net.fc = nn.Identity()
            self.backbone = net

        else:
            raise ValueError(f"Unsupported backbone: {backbone}")

        # -----------------------
        # Feature normalization (prevents spikes with different backbones)
        # -----------------------
        self.feat_norm = nn.LayerNorm(feat_dim)

        # -----------------------
        # Slice-level head
        # -----------------------
        self.head = nn.Sequential(
            nn.Linear(feat_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )

        # -----------------------
        # Load RadImageNet weights if provided
        # -----------------------
        if rad_weight_path is not None:
            ckpt = torch.load(rad_weight_path, map_location="cpu")
            sd = _clean_state_dict(ckpt)

            # Many RadImageNet checkpoints store keys under "backbone."
            sd = _strip_prefix(sd, "backbone.")

            # For inception checkpoints, sometimes AuxLogits keys exist; drop them if our model has none
            sd = {k: v for k, v in sd.items() if not k.startswith("AuxLogits.")}

            # ---- DenseNet121 fix: checkpoint uses "0.xxx" but torchvision expects "features.xxx" ----
            if backbone == "densenet121":
                if any(k.startswith("0.") for k in sd.keys()):
                    sd = {("features." + k[2:]) if k.startswith("0.") else k: v for k, v in sd.items()}

            # Load into backbone only
            missing, unexpected = self.backbone.load_state_dict(sd, strict=False)

            print(f"[RadImageNet] loaded weights from: {rad_weight_path}")
            print("  missing keys (first 10):", list(missing)[:10])
            print("  unexpected keys (first 10):", list(unexpected)[:10])

    def forward_slices(self, x):
        """
        x: [K,1,H,W] slices for one patient
        returns: [K] slice scores
        """
        x = self.in_proj(x)

        f = self.backbone(x)

        # --- stability guards ---
        f = torch.nan_to_num(f, nan=0.0, posinf=0.0, neginf=0.0)
        f = self.feat_norm(f)

        s = self.head(f).squeeze(-1)
        return s


def pool_scores(scores, mode="mean", topk=0.1, topk_k=None, mix=0.7):
    """
    scores: [K]
    mode: "mean", "topk", "mix"
    topk: fraction in (0,1] used if topk_k is None
    topk_k: fixed integer Ktop used if provided (recommended for representative sampling)
    mix: weight for topk in "mix" mode
    """
    if scores.numel() == 0:
        return torch.tensor(0.0, device=scores.device)

    if mode == "mean":
        return scores.mean()

    # decide k
    if topk_k is not None:
        k = int(topk_k)
    else:
        k = int(max(1, int(scores.numel() * float(topk))))

    k = max(1, min(k, scores.numel()))

    if mode == "topk":
        return scores.topk(k).values.mean()

    elif mode == "mix":
        top_part = scores.topk(k).values.mean()
        return float(mix) * top_part + (1.0 - float(mix)) * scores.mean()

    else:
        raise ValueError("mode must be 'mean', 'topk' or 'mix'")



In [ ]:
# Cell 11 - DataLoader collate: pack variable-K slice sets

from torch.utils.data import DataLoader

def collate_patient_bags(batch):
    """
    batch: list of (x, y, pid)
    x: [K,1,H,W]
    returns:
      xs: list of tensors, each [Ki,1,H,W]
      ys: tensor [B]
      pids: list[str]
    """
    xs, ys, pids = zip(*batch)
    ys = torch.stack(ys, dim=0)
    return list(xs), ys, list(pids)



In [ ]:
# Cell 12 - Training utilities: early stopping + evaluation metrics

import numpy as np
import torch
from scipy.stats import spearmanr

def eval_loader(model, loader, device, pool_mode="topk", topk_frac=0.2, topk_k=None, mix=0.7):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for xs, ys, pids in loader:
            ys = ys.to(device)
            preds = []
            for x in xs:
                x = x.to(device)
                scores = model.forward_slices(x)
                pred = pool_scores(scores, mode=pool_mode, topk=topk_frac, topk_k=topk_k, mix=mix)
                preds.append(pred)
            preds = torch.stack(preds)

            y_true.extend(ys.detach().cpu().numpy().tolist())
            y_pred.extend(preds.detach().cpu().numpy().tolist())

    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mae  = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred)**2)))

    # Spearman (can be noisy with n=7)
    rho = float(spearmanr(y_true, y_pred).correlation)

    # R^2 (can be negative; still report)
    sse = float(np.sum((y_true - y_pred) ** 2))
    sst = float(np.sum((y_true - np.mean(y_true)) ** 2))
    r2  = float(1.0 - sse / (sst + 1e-12))

    return {"MAE": mae, "RMSE": rmse, "Spearman": rho, "R2": r2}




In [ ]:
# Cell 13 — Flexible backbone (ImageNet/RadImageNet) + MSE + clipping + partial unfreeze + mild aug

from torch.utils.data import DataLoader
import torch
import numpy as np
import torchvision.transforms.functional as TF
from torch.cuda.amp import autocast, GradScaler

# ----------------------------
#   MODEL CONFIG
# ----------------------------
# options: "resnet18", "resnet50", "densenet121", "inceptionv3"
MODEL_CONFIG = {
    "backbone": "resnet18",
    "use_radimagenet": False,   # True for resnet50/densenet121/inceptionv3 since we have weights
}

def get_rad_path(backbone: str):
    if not MODEL_CONFIG["use_radimagenet"]:
        return None
    if backbone not in ["resnet50", "densenet121", "inceptionv3"]:
        raise ValueError("RadImageNet weights configured only for: resnet50, densenet121, inceptionv3")
    return RADIMAGENET_WEIGHTS[backbone]


# ----------------------------
#   Freezing helpers
# ----------------------------
def set_backbone_trainable(model, trainable: bool):
    for p in model.backbone.parameters():
        p.requires_grad = trainable

def set_partial_trainable(model, trainable: bool):
    bb = model.backbone

    # ResNet Sequential backbone (RadImageNet resnet50 case)
    if isinstance(bb, torch.nn.Sequential) and len(bb) >= 8:
        # index 7 is layer4
        for p in bb[7].parameters():
            p.requires_grad = trainable
        return

    # Regular torchvision ResNet
    if hasattr(bb, "layer4"):
        for p in bb.layer4.parameters():
            p.requires_grad = trainable
        return

    # DenseNet
    if hasattr(bb, "features") and hasattr(bb.features, "denseblock4"):
        for p in bb.features.denseblock4.parameters():
            p.requires_grad = trainable
        if hasattr(bb.features, "norm5"):
            for p in bb.features.norm5.parameters():
                p.requires_grad = trainable
        return

    # Inception late blocks
    mixed_names = ["Mixed_7a", "Mixed_7b", "Mixed_7c"]
    for name in mixed_names:
        if hasattr(bb, name):
            for p in getattr(bb, name).parameters():
                p.requires_grad = trainable
            return

    print("[warn] Unknown backbone structure for partial unfreeze; falling back to whole-backbone toggle.")
    set_backbone_trainable(model, trainable)


# ----------------------------
#    Augmentation (CT-safe)
# ----------------------------
def augment_ct_slice_batch(x, max_deg=5, max_shift=6, jitter=0.03, noise=0.02):
    """
    x: torch tensor [K,1,H,W] in [0,1]
    Applies same affine params to all slices (keeps geometry consistent).
    """
    angle = float(np.random.uniform(-max_deg, max_deg))
    tx = int(np.random.uniform(-max_shift, max_shift))
    ty = int(np.random.uniform(-max_shift, max_shift))
    scale = float(np.random.uniform(0.98, 1.02))

    out = []
    for i in range(x.shape[0]):
        img = x[i]  # [1,H,W]
        img = TF.affine(img, angle=angle, translate=[tx, ty], scale=scale, shear=[0.0, 0.0])
        out.append(img)
    x = torch.stack(out, dim=0)

    x = torch.clamp(
        x * float(np.random.uniform(1.0 - jitter, 1.0 + jitter)) + float(np.random.uniform(-jitter, jitter)),
        0.0, 1.0
    )
    x = torch.clamp(x + noise * torch.randn_like(x), 0.0, 1.0)
    return x


# ----------------------------
#    Training with ES
# ----------------------------
def train_fold_es(
    fold=0,
    pool_mode="topk",
    topk_frac=0.1,
    topk_k=None,
    mix=0.7,
    max_epochs=30,
    patience=4,
    lr=1e-4,
    lr_unfreeze=2e-4,
    batch_size=1,
    freeze_epochs=3,
    dropout=0.4,
    seed=42,
    grad_clip=1.0,
    wd=1e-4,
    use_aug=True,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    index_csv = f"{OUT_ROOT}/train_index.csv"
    train_ds = PatientSliceDataset(index_csv=index_csv, fold=fold, split="train", seed=seed)
    val_ds   = PatientSliceDataset(index_csv=index_csv, fold=fold, split="val",   seed=seed)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0,
                              collate_fn=collate_patient_bags)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0,
                              collate_fn=collate_patient_bags)

    backbone = MODEL_CONFIG["backbone"]
    rad_path = get_rad_path(backbone)

    # IMPORTANT: this expects you to be using the UPDATED Cell 11 SliceRegressor signature:
    # SliceRegressor(backbone=..., imagenet_pretrained=..., rad_weight_path=...)
    model = SliceRegressor(
        backbone=backbone,
        imagenet_pretrained=(rad_path is None),
        rad_weight_path=rad_path
    ).to(device)

    # set dropout in head
    if isinstance(model.head, torch.nn.Sequential):
        for m in model.head.modules():
            if isinstance(m, torch.nn.Dropout):
                m.p = dropout

    # freeze entire backbone first
    set_backbone_trainable(model, False)
    for p in model.head.parameters():
        p.requires_grad = True
    for p in model.in_proj.parameters():
        p.requires_grad = True

    loss_fn = torch.nn.MSELoss()

    opt = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                            lr=lr, weight_decay=wd)

    best = {"epoch": -1, "RMSE": np.inf, "state": None, "metrics": None}
    bad_epochs = 0

    for ep in range(1, max_epochs + 1):

        # after freeze_epochs: unfreeze last block only
        if ep == freeze_epochs + 1:
            set_partial_trainable(model, True)
            opt = torch.optim.AdamW(
                filter(lambda p: p.requires_grad, model.parameters()),
                                    lr=lr*0.3, weight_decay=wd)

        model.train()
        tr_losses = []

        for xs, ys, _ in train_loader:
            ys = ys.to(device)

            preds = []
            for x in xs:
                x = x.to(device)
                if use_aug:
                    x = augment_ct_slice_batch(x)

                scores = model.forward_slices(x)
                pred = pool_scores(scores, mode=pool_mode, topk=topk_frac, topk_k=topk_k, mix=mix)
                preds.append(pred)

            preds = torch.stack(preds)
            loss = loss_fn(preds, ys)

            opt.zero_grad()
            loss.backward()

            if grad_clip is not None and grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

            opt.step()
            tr_losses.append(float(loss.item()))

        val_metrics = eval_loader(model,
                                  val_loader,
                                  device,
                                  pool_mode=pool_mode,
                                  topk_frac=topk_frac,
                                  topk_k=topk_k,
                                  mix=mix)

        print(
            f"Fold {fold} | Epoch {ep:02d} | train MSE {np.mean(tr_losses):.4f} | "
            f"val MAE {val_metrics['MAE']:.3f} | val RMSE {val_metrics['RMSE']:.3f} | "
            f"val R2 {val_metrics['R2']:.3f} | val Spearman {val_metrics['Spearman']:.3f}"
        )

        if val_metrics["RMSE"] + 1e-6 < best["RMSE"]:
            best = {
                "epoch": ep,
                "RMSE": val_metrics["RMSE"],
                "state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "metrics": val_metrics,
            }
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= patience:
            print(f"Early stop: no RMSE improvement for {patience} epochs. "
                  f"Best epoch={best['epoch']} RMSE={best['RMSE']:.3f}")
            break

    if best["state"] is not None:
        model.load_state_dict(best["state"], strict=True)

    return best["metrics"], best["epoch"]



In [ ]:
# Cell 14 - Run full 5-fold CV and summarize

results = []
best_epochs = []

for fold in range(5):
    metrics, best_ep = train_fold_es(
        fold=fold,
        pool_mode="mix",
        mix=0.7,
        topk_k=10,        # fixed topk
        topk_frac=0.2,    # ignored because topk_k is set, but keep for compatibility
        max_epochs=30,
        patience=7,
        lr=1e-4,
        batch_size=2,
        freeze_epochs=3,
        dropout=0.3,
        seed=42
    )
    results.append({"fold": fold, **metrics})
    best_epochs.append(best_ep)
    print("Fold", fold, "best epoch:", best_ep, "metrics:", metrics)

df_res = pd.DataFrame(results)
print("\nPer-fold results:")
display(df_res)

print("\nSummary (mean ± std):")
for col in ["MAE","RMSE","R2","Spearman"]:
    m = df_res[col].mean()
    s = df_res[col].std(ddof=1)
    print(f"{col}: {m:.3f} ± {s:.3f}")

print("\nBest epochs per fold:", best_epochs)

